# Знакомство с разметкой данных

- Ниже **демонстрационный код**, который позволит познакомиться с несколькими типами задач разметки данных на практике. Он не заменяет профессиональные инструменты разметки (Label Studio, CVAT и т.п.), но наглядно показывает **принципы**.
- Все данные (разметка) сохраняются в CSV-файлы в папке results в папке лабораторной работы 4. 
- Вы можете сможете самостоятельно выполнить несколько примеров, а также поменять исходные данные (тексты, изображения, аудио-ссылки) прямо в коде, чтобы экспериментировать.

Запускайте ячейки последовательно (Shift+Enter), в каждой демонстрации следуйте инструкциям.

In [ ]:
import os
import io
import requests
import ipywidgets as widgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, IntSlider, fixed
from IPython.display import Audio, display, clear_output
from PIL import Image

In [ ]:
# Проверка папки для хранения результатов
os.makedirs("results", exist_ok=True)

# Разметка текста (классификация тональности)

Запустите ячейку и проставьте класс для текста

In [ ]:
# Исходные тексты
texts = [
    "Визуальный ряд гипнотизирует, а саундтрек пробирает до мурашек — безусловный шедевр.",
    "Советую смотреть тем, кто любит дыры в сюжете",
    "Книга на 407 страниц про пса была довольно неплохая, но есть к чему стремиться.",
    "Абсолютная магия на экране! Уже купил билет на повторный сеанс на круглый хлеб",
    "Смотреть не буду, дождусь бэдкомедиана"
]

# Хранилище результатов
results = []
current_idx = 0

# Виджеты
text_display = widgets.HTML(value="<h3>Текст:</h3><p>Загрузка...</p>")
btn_pos = widgets.Button(description="😊 Позитив", button_style='success')
btn_neg = widgets.Button(description="😞 Негатив", button_style='danger')
btn_neu = widgets.Button(description="😐 Нейтрально", button_style='warning')
btn_skip = widgets.Button(description="⏭ Пропустить", button_style='info')
status = widgets.HTML(value="Прогресс: 0 / {}".format(len(texts)))

output = widgets.Output()

def show_next():
    global current_idx
    clear_output(wait=True)
    if current_idx < len(texts):
        text_display.value = f"<h3>Текст {current_idx+1}/{len(texts)}</h3><p>{texts[current_idx]}</p>"
        status.value = f"Прогресс: {len(results)} / {len(texts)}"
        display(widgets.VBox([text_display, widgets.HBox([btn_pos, btn_neg, btn_neu, btn_skip]), status]))
    else:
        # Завершение
        df = pd.DataFrame(results, columns=["text", "label"])
        display(df)
        df.to_csv("results/labeled_texts.csv", index=False)
        print("Разметка завершена. Данные сохранены в labeled_texts.csv")
        display(widgets.HTML("<b>Статистика:</b>"))
        display(df["label"].value_counts())

def save_label(label):
    global current_idx
    if current_idx < len(texts):
        results.append({"text": texts[current_idx], "label": label})
        current_idx += 1
        show_next()

btn_pos.on_click(lambda b: save_label("Позитив"))
btn_neg.on_click(lambda b: save_label("Негатив"))
btn_neu.on_click(lambda b: save_label("Нейтрально"))
btn_skip.on_click(lambda b: save_label("Пропущено"))

# Запуск
show_next()

# Разметка изображений

In [ ]:
# Загрузка изображения
# url = "https://www.contlease.ru/upload/medialibrary/d09/d09398c59f1bab2153fc23f6a5357821.jpg"
url = "https://avatars.mds.yandex.net/i?id=354885f24ab884bebdc97c38b778967ffa7602fb-5552883-images-thumbs&n=13"
response = requests.get(url)
img = Image.open(io.BytesIO(response.content))
img = img.resize((400, 300))
img_array = np.array(img)

In [ ]:
# Виджеты-слайдеры
x_slider = IntSlider(min=0, max=img.width, step=1, value=50, description='X')
y_slider = IntSlider(min=0, max=img.height, step=1, value=50, description='Y')
w_slider = IntSlider(min=10, max=img.width, step=1, value=100, description='Width')
h_slider = IntSlider(min=10, max=img.height, step=1, value=100, description='Height')

# Кнопка сохранения координат
save_btn = widgets.Button(description="💾 Сохранить рамку", button_style='primary')
coords_output = widgets.Output()

def draw_bbox(x, y, w, h):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(img_array)
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(f"Bounding Box: x={x}, y={y}, w={w}, h={h}")
    plt.close(fig)  # чтобы не дублировать в выводе
    return fig

# Интерактивное обновление
interact(draw_bbox, x=x_slider, y=y_slider, w=w_slider, h=h_slider);

# Сохранение координат
def on_save(b):
    with coords_output:
        clear_output()
        coord = {
            'x': x_slider.value,
            'y': y_slider.value,
            'width': w_slider.value,
            'height': h_slider.value
        }
        # Можно добавить в список или записать в файл
        print(f"Сохранены координаты: {coord}")
        # Пример записи в CSV (добавляем строку)
        df = pd.DataFrame([coord])
        if not hasattr(on_save, 'df'):
            on_save.df = df
        else:
            on_save.df = pd.concat([on_save.df, df], ignore_index=True)
        on_save.df.to_csv("results/bboxes.csv", index=False)
        print("Текущий файл bboxes.csv обновлён")

save_btn.on_click(on_save)
display(save_btn, coords_output)

# Разметка аудио

In [ ]:
# Список URL-адресов аудиофрагментов
audio_urls = [
    "https://zvukitop.com/wp-content/uploads/567.mp3",  
    "https://zvukitop.com/wp-content/uploads/554.mp3",
    "https://zvukitop.com/wp-content/uploads/2021/04/probka-rwsr.mp3",
    "https://zvukitop.com/wp-content/uploads/two-hours-later.mp3",
]

# Варианты для классификации
categories = ["Речь", "Музыка", "Шум", "Животные", "Другое"]

In [ ]:
results = []
current_idx = 0

# Виджеты
audio_output = widgets.Output()
dropdown = widgets.Dropdown(
    options=["— Выберите категорию —"] + categories,
    description="Категория"
)
save_btn = widgets.Button(description="💾 Сохранить и далее", button_style='primary')
skip_btn = widgets.Button(description="⏭ Пропустить", button_style='warning')
status_label = widgets.HTML(value="")
progress_bar = widgets.IntProgress(
    value=0, min=0, max=len(audio_urls), description='Прогресс'
)

def show_audio(index):
    with audio_output:
        clear_output(wait=True)
        if index < len(audio_urls):
            url = audio_urls[index]
            # Показываем плеер
            display(Audio(url, autoplay=False))
            status_label.value = f"🎵 Фрагмент {index+1} из {len(audio_urls)}"
            progress_bar.value = index
        else:
            status_label.value = "✅ Разметка завершена!"
            progress_bar.value = len(audio_urls)

def save_choice(label):
    global current_idx
    if current_idx < len(audio_urls):
        results.append({
            "url": audio_urls[current_idx],
            "label": label
        })
        current_idx += 1
        dropdown.value = "— Выберите категорию —"
        if current_idx < len(audio_urls):
            show_audio(current_idx)
        else:
            show_audio(current_idx)
            # Сохраняем итоговый DataFrame
            df = pd.DataFrame(results)
            display(df)
            df.to_csv("results/audio_labels.csv", index=False, encoding='utf-8')
            print("Все фрагменты размечены. Данные сохранены в audio_labels.csv")
    else:
        print("Все фрагменты уже обработаны.")

def on_save(b):
    if dropdown.value == "— Выберите категорию —":
        print("Пожалуйста, выберите категорию перед сохранением!")
        return
    save_choice(dropdown.value)

def on_skip(b):
    if current_idx < len(audio_urls):
        save_choice("Пропущено")
    else:
        print("Нет фрагментов для пропуска.")

save_btn.on_click(on_save)
skip_btn.on_click(on_skip)

# Запуск, если есть хотя бы одна ссылка
if audio_urls:
    show_audio(0)
    display(
        widgets.VBox([
            audio_output,
            widgets.HBox([dropdown, save_btn, skip_btn]),
            status_label,
            progress_bar
        ])
    )
else:
    print("Список audio_urls пуст. Добавьте хотя бы одну ссылку.")